In [1]:
%%capture
!git clone https://github.com/Phan-Trung-Thuan/german-VSL /kaggle/working/german-VSL
%cd /kaggle/working/german-VSL
!python kaggle_setup.py

In [12]:
# Create a working directory for annotations
!mkdir -p /kaggle/working/phoenix_annotations

# Copy the compressed files from the input directory to the working directory
!cp /kaggle/input/datasets/mariusschmidtmengin/phoenixweather2014t-3rd-attempt/*.gzip /kaggle/working/phoenix_annotations/

# Unzip the files 
!gunzip -S .gzip /kaggle/working/phoenix_annotations/*.gzip

In [16]:
import os
import gzip
import pickle
from torch.utils.data import Dataset

class Phoenix14TDataset(Dataset):
    def __init__(self, annotation_file, video_dir):
        """
        annotation_file: Path to the .gz annotation file
        video_dir: Path to the specific split folder containing the videos
        """
        self.video_dir = video_dir
        
        # Open and decompress the file directly in memory
        with gzip.open(annotation_file, 'rb') as f:
            raw_data = pickle.load(f)
            
            # PHOENIX annotations are typically a list of dictionaries. 
            # If they are stored as a dictionary mapping, we convert to a list.
            if isinstance(raw_data, dict):
                self.annotations = list(raw_data.values())
            else:
                self.annotations = raw_data

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        # Extract the specific annotation mapping
        anno = self.annotations[idx]
        
        # Retrieve the video name (commonly stored under the 'name' or 'folder' key)
        video_name = anno.get('name', str(idx))
        
        # Create the full video path (Assuming Kaggle mp4 format)
        video_path = os.path.join(self.video_dir, f"{video_name}.mp4")
        
        # Return input (video path) and output (annotation dict)
        return video_path, anno


# --- Configuration & Instantiation ---
BASE_PATH = "//kaggle/input/datasets/mariusschmidtmengin/phoenixweather2014t-3rd-attempt"

train_anno = os.path.join(BASE_PATH, "phoenix14t.pami0.train.annotations_only.gzip")
train_vids = os.path.join(BASE_PATH, "videos_phoenix", "videos", "train")

test_anno = os.path.join(BASE_PATH, "phoenix14t.pami0.test.annotations_only.gzip")
test_vids = os.path.join(BASE_PATH, "videos_phoenix", "videos", "test")

# Create the datasets
trainset = Phoenix14TDataset(train_anno, train_vids)
testset = Phoenix14TDataset(test_anno, test_vids)

# --- Output the requirements ---

# Count examples in each set
print(f"Total training examples: {len(trainset)}")
print(f"Total testing examples: {len(testset)}\n")

# Print out exactly 1 example from the trainset
sample_video_path, sample_annotation = trainset[0]

print("--- 1 Example from Trainset ---")
print(f"Input Video Path : {sample_video_path}")
print(f"Output Annotation: {sample_annotation}")

Total training examples: 7096
Total testing examples: 642

--- 1 Example from Trainset ---
Input Video Path : //kaggle/input/datasets/mariusschmidtmengin/phoenixweather2014t-3rd-attempt/videos_phoenix/videos/train/train/11August_2010_Wednesday_tagesschau-1.mp4
Output Annotation: {'name': 'train/11August_2010_Wednesday_tagesschau-1', 'signer': 'Signer08', 'gloss': 'JETZT WETTER MORGEN DONNERSTAG ZWOELF FEBRUAR', 'text': 'und nun die wettervorhersage für morgen donnerstag den zwölften august .'}


---
## Research Pipeline

5 independent modules — each output is passed to the next:

| Module | Input | Output |
|--------|-------|--------|
| `module1` | video path | AudioResult (16-kHz WAV) |
| `module2` | AudioResult | ASRResult (German text) |
| `module3` | ASRResult | GlossResult (gloss tokens) |
| `module4` | GlossResult + lexicon | LookupResult (video clips) |
| `module5` | LookupResult | GIFResult (animated GIF) |

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/german-VSL')

from pipeline import (
    PhoenixLexicon,
    module1, module2, module3, module4, module5,
    display_gif, run_pipeline, evaluate_example,
)

print('Pipeline modules loaded OK')

In [ ]:
# Build PhoenixLexicon directly from the trainset already loaded above.
# This scans all training annotations and indexes each gloss token
# to the video(s) that contain it -- no separate CSV file required.

BASE_PATH  = '//kaggle/input/datasets/mariusschmidtmengin/phoenixweather2014t-3rd-attempt'
VIDEOS_DIR = f'{BASE_PATH}/videos_phoenix/videos'

lex = PhoenixLexicon.from_dataset(
    dataset    = trainset,
    videos_dir = VIDEOS_DIR,
    split      = 'train',
)
print(lex.stats())

In [ ]:
# Reuse the example already printed by the dataset cell above
video_path = sample_video_path
annotation = sample_annotation

print('Video path :', video_path)
print('Gloss (GT) :', annotation['gloss'])
print('Text  (GT) :', annotation['text'])

In [ ]:
# Module 1 -- Video -> Audio
o1 = module1(video_path)
print(o1)

In [ ]:
# Module 2 -- Audio -> German Text  (ASR)
# Canary-1B is cached after first call (~4 GB download on first run).
o2 = module2(o1)
print(o2)

In [ ]:
# Module 3 -- German Text -> Gloss Tokens
o3 = module3(o2)
print(o3)
print()
print('Ground-truth gloss :', annotation['gloss'])
print('Predicted gloss    :', ' '.join(o3.gloss_tokens))

In [ ]:
# Module 4 -- Gloss Tokens -> Video Clips  (Phoenix14T lexicon lookup)
o4 = module4(o3, lex)
print(o4)

In [ ]:
# Module 5 -- Video Clips -> Animated GIF
o5 = module5(o4, 'output.gif')
print(o5)

display_gif('output.gif')

In [ ]:
# Compare predicted gloss tokens to the Phoenix14T ground truth
example = {
    'video_path': video_path,
    'name'      : annotation['name'],
    'gloss'     : annotation['gloss'],
    'text'      : annotation['text'],
}

metrics = evaluate_example(example, lex)

print('Ground-truth text  :', metrics['gt_text'])
print('Predicted text     :', metrics['pred_text'])
print()
print('Ground-truth gloss :', metrics['gt_gloss'])
print('Predicted gloss    :', metrics['pred_gloss'])
print()
print(f"Precision : {metrics['precision']:.3f}")
print(f"Recall    : {metrics['recall']:.3f}")
print(f"F1        : {metrics['f1']:.3f}")
print(f"Gloss WER : {metrics['gloss_wer']:.3f}")

In [ ]:
# (Optional) Run full pipeline in one call
results = run_pipeline(
    video_path = video_path,
    lexicon    = lex,
    output_gif = 'output_full.gif',
)
display_gif('output_full.gif')